# SpectralBridge bulk production workflow — FINAL

## One notebook, one Run All, complete production result

This notebook is the consolidated production workflow for SpectralBridge bulk
cross-sensor analysis.

A successful run means **all** of the following completed:

- source discovery and minimal transfer
- duplicate-flightline reconciliation
- scientific preflight
- restart-safe population analysis
- authoritative candidate coefficients and sufficient statistics
- compact scientific interpretation via the current `summarize_bulk_results()` API
- QA dashboard and diagnostics
- publication figures
- Markdown and PDF reports
- complete closeout packaging
- SHA256 preservation audit
- optional CyVerse upload and remote verification

There is no "summary unavailable but continue" path. If the current SpectralBridge
checkout cannot produce the scientific interpretation/report products, the notebook
fails rather than declaring production success.


In [ ]:

# =====================================================================
# 0. USER CONFIGURATION
# =====================================================================

from pathlib import Path

BASE = Path("/home/jovyan/data-store").resolve()

REMOTE_SOURCE = "/iplant/home/shared/earthlab/macrosystems/YOUR_CURATED_COLLECTION"
RUN_NAME = Path(REMOTE_SOURCE.rstrip("/")).name

# Allowed:
# "discover", "plan", "transfer", "reconcile", "preflight",
# "analyze", "summarize", "package", "upload", "verify", "all"
RUN_STAGE = "all"
RUN = False  # Explicitly enable only after reviewing source, disk, and output paths.
ALLOW_REBUILD_RECONCILED_STAGE = False
ALLOW_REBUILD_PACKAGE = False

# Remote writes are explicit. Set True only when you want the final package uploaded.
UPLOAD_RESULTS = False

DISK_SAFETY_RESERVE_GB = 80
MAX_STAGE_FRACTION_OF_FREE = 0.75

MAX_REMOTE_COLLECTIONS = 5000
MAX_REMOTE_DEPTH = 8

EXTRACTION_WORKERS = 1
FIGURE_DPI = 220

# Raw downloaded candidate products.
LOCAL_STAGE = BASE / f"{RUN_NAME}_Minimal"

# Scientifically reconciled unique-flightline stage.
RECONCILED_STAGE = BASE / f"{RUN_NAME}_Minimal_Reconciled"

PREFLIGHT_OUTPUT = BASE / f"{RUN_NAME}_Bulk_Preflight"
FINAL_OUTPUT = BASE / f"{RUN_NAME}_Bulk_Analysis"
STATE_DIR = BASE / f"{RUN_NAME}_Bulk_State"

PACKAGE_ROOT = BASE / "SpectralBridge_Bulk_Closeout"
PACKAGE_DIR = PACKAGE_ROOT / f"SpectralBridge_{RUN_NAME}_Bulk_Analysis"

TEMP_DIR = Path("/home/jovyan/work") / f"spectralbridge_{RUN_NAME.lower()}_tmp"

REMOTE_SOURCE_URI = "i:" + REMOTE_SOURCE
GOCMD = "gocmd"

# Hash only repeated scientific units. This is deliberate: duplicate reconciliation
# should establish content identity, not infer it from file size alone.
HASH_DUPLICATE_GROUPS = True

ALLOWED_STAGES = {
    "discover",
    "plan",
    "transfer",
    "reconcile",
    "preflight",
    "analyze",
    "summarize",
    "package",
    "upload",
    "verify",
    "all",
}

if RUN_STAGE not in ALLOWED_STAGES:
    raise ValueError(f"RUN_STAGE must be one of {sorted(ALLOWED_STAGES)}")
if not RUN:
    raise RuntimeError("Review the configuration, then set RUN = True to start this production notebook.")
if "YOUR_CURATED_COLLECTION" in REMOTE_SOURCE:
    raise ValueError("Replace REMOTE_SOURCE with the curated collection you intend to analyze.")

print("RUN NAME          :", RUN_NAME)
print("RUN STAGE         :", RUN_STAGE)
print("REMOTE SOURCE     :", REMOTE_SOURCE)
print("RAW LOCAL STAGE   :", LOCAL_STAGE)
print("RECONCILED STAGE  :", RECONCILED_STAGE)
print("PREFLIGHT         :", PREFLIGHT_OUTPUT)
print("FINAL OUTPUT      :", FINAL_OUTPUT)
print("STATE DIR         :", STATE_DIR)
print("PACKAGE           :", PACKAGE_DIR)



## Current execution mode

`RUN = False` is the safe published default. Replace `REMOTE_SOURCE` and review
all output paths and disk limits before setting `RUN = True`.

With `RUN = True`, Run All performs the full production workflow; remote upload
also requires `UPLOAD_RESULTS = True` and every preceding gate to pass.
The reconciliation gate will stop the run
if repeated flightline copies differ in size or content.

The notebook never modifies or deletes the CyVerse source collection.


## 1. Common helpers and repository discovery

In [ ]:

import os
import sys
import re
import json
import shutil
import hashlib
import subprocess
import importlib
import inspect
import pkgutil
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

STATE_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(args, cwd=None):
    return subprocess.run(
        [str(x) for x in args],
        text=True,
        capture_output=True,
        cwd=cwd,
        check=False,
    )

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def require_file(path, label=None):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Required file missing or empty: {label or path}")
    return path

def require_dir(path, label=None):
    path = Path(path)
    if not path.is_dir():
        raise RuntimeError(f"Required directory missing: {label or path}")
    return path

def assert_within(path, parent):
    path = Path(path).resolve()
    parent = Path(parent).resolve()
    if path == parent or parent in path.parents:
        return path
    raise RuntimeError(f"Path escapes allowed parent: {path}")

def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def copy_file(src, dst):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

if shutil.which(GOCMD) is None:
    raise RuntimeError("gocmd is not available on this VM.")

repo_candidates = [
    BASE / "spectralbridge",
    Path("/home/jovyan/spectralbridge"),
    Path.cwd(),
]

repo_root = next(
    (
        p.resolve()
        for p in repo_candidates
        if (p / ".git").is_dir()
        and (p / "src/spectralbridge").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("Local SpectralBridge repository not found.")

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

importlib.invalidate_caches()
import spectralbridge
try:
    from spectralbridge import summarize_bulk_results
    from spectralbridge.bulk import BulkResultsConfig
except ImportError as exc:
    raise RuntimeError(
        "This production notebook requires the current SpectralBridge bulk-results "
        "API: spectralbridge.summarize_bulk_results and "
        "spectralbridge.bulk.BulkResultsConfig. Update the local checkout before "
        "running production."
    ) from exc

git_info = {}
for key, cmd in {
    "commit": ["git", "rev-parse", "HEAD"],
    "branch": ["git", "branch", "--show-current"],
    "status": ["git", "status", "--short"],
    "origin": ["git", "remote", "get-url", "origin"],
}.items():
    res = run_cmd(cmd, cwd=repo_root)
    git_info[key] = (res.stdout or "").strip()

print("Python:", sys.version.split()[0])
print("gocmd:", shutil.which(GOCMD))
print("SpectralBridge version:", getattr(spectralbridge, "__version__", "unknown"))
print("Repository:", repo_root)
display(pd.DataFrame([git_info]))



## 2. DISCOVER stage: locate the actual package API

This scans the checked-out `spectralbridge` package for the required production
objects instead of assuming their module locations.

Required:
- `run_bulk_pipeline`
- `DEFAULT_PRODUCT_REGISTRY`

Also required for the complete production closeout:
- `summarize_bulk_results`
- `BulkResultsConfig`

If compact summarization is unavailable or incompatible, this notebook
stops before starting the expensive production analysis.


In [ ]:

API_DISCOVERY_JSON = STATE_DIR / "api_discovery.json"

def discover_symbol(symbol_name):
    hits = []

    # Top-level package first.
    if hasattr(spectralbridge, symbol_name):
        hits.append({
            "module": "spectralbridge",
            "symbol": symbol_name,
        })

    package_path = Path(spectralbridge.__file__).resolve().parent

    for mod_info in pkgutil.walk_packages(
        [str(package_path)],
        prefix="spectralbridge.",
    ):
        module_name = mod_info.name

        # Avoid importing test modules or obviously unrelated examples.
        if ".tests" in module_name or module_name.endswith(".tests"):
            continue

        try:
            module = importlib.import_module(module_name)
        except Exception:
            continue

        if hasattr(module, symbol_name):
            hits.append({
                "module": module_name,
                "symbol": symbol_name,
            })

    # Deduplicate while preserving order.
    seen = set()
    unique = []
    for hit in hits:
        key = (hit["module"], hit["symbol"])
        if key not in seen:
            seen.add(key)
            unique.append(hit)

    return unique

if RUN_STAGE not in ("discover", "all"):
    print("DISCOVER stage skipped.")
else:
    symbols = {
        "run_bulk_pipeline": discover_symbol("run_bulk_pipeline"),
        "DEFAULT_PRODUCT_REGISTRY": discover_symbol("DEFAULT_PRODUCT_REGISTRY"),
        "summarize_bulk_results": discover_symbol("summarize_bulk_results"),
        "BulkResultsConfig": discover_symbol("BulkResultsConfig"),
    }

    print(json.dumps(symbols, indent=2))

    if not symbols["run_bulk_pipeline"]:
        raise RuntimeError("Could not discover run_bulk_pipeline in this checkout.")

    if not symbols["DEFAULT_PRODUCT_REGISTRY"]:
        raise RuntimeError("Could not discover DEFAULT_PRODUCT_REGISTRY in this checkout.")

    # Pick first discovered implementation, but record every candidate.
    run_bulk_module = symbols["run_bulk_pipeline"][0]["module"]
    registry_module = symbols["DEFAULT_PRODUCT_REGISTRY"][0]["module"]

    run_bulk_pipeline = getattr(
        importlib.import_module(run_bulk_module),
        "run_bulk_pipeline",
    )
    DEFAULT_PRODUCT_REGISTRY = getattr(
        importlib.import_module(registry_module),
        "DEFAULT_PRODUCT_REGISTRY",
    )

    bulk_params = inspect.signature(run_bulk_pipeline).parameters
    required_bulk = {
        "input_path",
        "output_dir",
        "input_mode",
        "analysis",
        "on_invalid",
        "materialize_observations",
        "preflight_only",
        "extraction_workers",
        "temp_directory",
    }

    missing_bulk = sorted(required_bulk - set(bulk_params))
    if missing_bulk:
        raise RuntimeError(
            "Discovered run_bulk_pipeline is incompatible with this production "
            f"notebook. Missing parameters: {missing_bulk}"
        )

    summary_supported = bool(
        symbols["summarize_bulk_results"]
        and symbols["BulkResultsConfig"]
    )

    summary_record = None

    if summary_supported:
        summary_module = symbols["summarize_bulk_results"][0]["module"]
        config_module = symbols["BulkResultsConfig"][0]["module"]

        summarize_bulk_results = getattr(
            importlib.import_module(summary_module),
            "summarize_bulk_results",
        )
        BulkResultsConfig = getattr(
            importlib.import_module(config_module),
            "BulkResultsConfig",
        )

        summary_params = inspect.signature(summarize_bulk_results).parameters
        required_summary = {
            "bulk_output",
            "config",
            "make_figures",
            "make_report",
        }
        missing_summary = sorted(required_summary - set(summary_params))

        if missing_summary:
            summary_supported = False
            summary_record = {
                "reason": "signature_mismatch",
                "missing_parameters": missing_summary,
            }

    if not summary_supported:
        raise RuntimeError("Complete production closeout requires the current bulk results API.")

    record = {
        "checked_utc": datetime.now(timezone.utc).isoformat(),
        "spectralbridge_version": getattr(spectralbridge, "__version__", "unknown"),
        "repo": str(repo_root),
        "git": git_info,
        "symbols": symbols,
        "run_bulk_module": run_bulk_module,
        "registry_module": registry_module,
        "run_bulk_signature": str(inspect.signature(run_bulk_pipeline)),
        "summary_supported": bool(summary_supported),
        "summary_record": summary_record,
    }

    if summary_supported:
        record["summarize_module"] = symbols["summarize_bulk_results"][0]["module"]
        record["config_module"] = symbols["BulkResultsConfig"][0]["module"]
        record["summarize_signature"] = str(inspect.signature(summarize_bulk_results))

    write_json(API_DISCOVERY_JSON, record)

    print("\n✓ API DISCOVERY APPROVED.")
    print("run_bulk_pipeline:", record["run_bulk_module"])
    print("DEFAULT_PRODUCT_REGISTRY:", record["registry_module"])
    print("summary_supported:", record["summary_supported"])
    print("Next: planning will run automatically in RUN_STAGE='all' mode.")


## 3. Load discovered API for all later stages

In [ ]:

api = read_json(require_file(API_DISCOVERY_JSON))

run_bulk_pipeline = getattr(
    importlib.import_module(api["run_bulk_module"]),
    "run_bulk_pipeline",
)

DEFAULT_PRODUCT_REGISTRY = getattr(
    importlib.import_module(api["registry_module"]),
    "DEFAULT_PRODUCT_REGISTRY",
)

SUMMARY_SUPPORTED = bool(api.get("summary_supported"))

summarize_bulk_results = None
BulkResultsConfig = None

if SUMMARY_SUPPORTED:
    summarize_bulk_results = getattr(
        importlib.import_module(api["summarize_module"]),
        "summarize_bulk_results",
    )
    BulkResultsConfig = getattr(
        importlib.import_module(api["config_module"]),
        "BulkResultsConfig",
    )

print("Loaded production API from persisted discovery.")
print("Summary supported:", SUMMARY_SUPPORTED)


## 4. Persisted-state paths

In [ ]:

REMOTE_INVENTORY_CSV = STATE_DIR / "remote_inventory.csv"
FLIGHTLINE_AUDIT_CSV = STATE_DIR / "flightline_completeness.csv"
TRANSFER_MANIFEST_CSV = STATE_DIR / "transfer_manifest.csv"
STAGE_VALIDATION_CSV = STATE_DIR / "stage_validation.csv"

DUPLICATE_AUDIT_CSV = STATE_DIR / "duplicate_resolution.csv"
RECONCILED_MANIFEST_CSV = STATE_DIR / "reconciled_manifest.csv"
RECONCILIATION_JSON = STATE_DIR / "reconciliation.json"

PLAN_JSON = STATE_DIR / "plan.json"
PREFLIGHT_GATE_JSON = STATE_DIR / "preflight_gate.json"
ANALYSIS_GATE_JSON = STATE_DIR / "analysis_gate.json"
SUMMARY_GATE_JSON = STATE_DIR / "summary_gate.json"
PACKAGE_GATE_JSON = STATE_DIR / "package_gate.json"
UPLOAD_GATE_JSON = STATE_DIR / "upload_gate.json"

print("State directory:", STATE_DIR)


## 5. PLAN stage: verify and inventory the exact CyVerse source

In [ ]:

if RUN_STAGE not in ("plan", "all"):
    print("PLAN inventory skipped.")
else:
    probe = run_cmd([GOCMD, "ls", REMOTE_SOURCE_URI])

    if probe.returncode != 0:
        print(probe.stderr or "")
        raise RuntimeError("Cannot access the requested CyVerse source collection.")

    collection_re = re.compile(r"^\s*collection\s+(\S+)")
    data_re = re.compile(r"^\s*data-object\s+(\S+)\s+(\d+)\b")

    def parse_gocmd_ls(text):
        children, objects = [], []

        for raw in text.splitlines():
            line = raw.strip()
            if not line:
                continue

            m = collection_re.match(line)
            if m:
                name = m.group(1)
                if not name.startswith("/"):
                    children.append(name)
                continue

            m = data_re.match(line)
            if m:
                objects.append((m.group(1), int(m.group(2))))

        return children, objects

    queue = [(REMOTE_SOURCE, 0)]
    visited = set()
    rows = []
    listing_errors = []
    depth_truncated = []

    while queue:
        if len(visited) >= MAX_REMOTE_COLLECTIONS:
            raise RuntimeError(
                f"Inventory exceeded MAX_REMOTE_COLLECTIONS={MAX_REMOTE_COLLECTIONS}."
            )

        current, depth = queue.pop(0)

        if current in visited:
            continue
        visited.add(current)

        res = run_cmd([GOCMD, "ls", "i:" + current])

        if res.returncode != 0:
            listing_errors.append({
                "collection": current,
                "returncode": res.returncode,
                "stderr": (res.stderr or "")[-1200:],
            })
            continue

        children, objects = parse_gocmd_ls(res.stdout or "")

        for name, size in objects:
            rows.append({
                "type": "data-object",
                "remote_path": current.rstrip("/") + "/" + name,
                "bytes": int(size),
                "depth": depth,
            })

        if children and depth >= MAX_REMOTE_DEPTH:
            depth_truncated.append({
                "collection": current,
                "depth": depth,
                "child_count": len(children),
            })
        else:
            for child in children:
                queue.append((current.rstrip("/") + "/" + child, depth + 1))

    if listing_errors:
        display(pd.DataFrame(listing_errors).head(100))
        raise RuntimeError("Remote inventory is incomplete.")

    if depth_truncated:
        display(pd.DataFrame(depth_truncated).head(100))
        raise RuntimeError("Remote inventory hit MAX_REMOTE_DEPTH.")

    remote_inventory = pd.DataFrame(rows)

    if remote_inventory.empty:
        raise RuntimeError("No remote data objects were discovered.")

    remote_inventory.to_csv(REMOTE_INVENTORY_CSV, index=False)

    print("✓ Remote inventory complete.")
    print("Collections visited:", len(visited))
    print("Objects found:", len(remote_inventory))
    display(remote_inventory.head(100))


## 6. PLAN stage: derive the minimal scientifically complete archive

In [ ]:

if RUN_STAGE not in ("plan", "all"):
    print("PLAN archive derivation skipped.")
else:
    remote_inventory = pd.read_csv(require_file(REMOTE_INVENTORY_CSV))

    objects = remote_inventory.copy()
    objects["name"] = objects["remote_path"].map(lambda x: Path(x).name)
    objects["parent"] = objects["remote_path"].map(lambda x: str(Path(x).parent))
    objects["suffix"] = objects["name"].map(lambda x: Path(x).suffix.lower())

    translation_pairs = tuple(DEFAULT_PRODUCT_REGISTRY.translation_pairs)

    required_sensors = sorted({
        sensor
        for pair in translation_pairs
        for sensor in (pair.source_sensor, pair.target_sensor)
    })

    recognized_rows = []

    for _, row in objects[objects["suffix"] == ".img"].iterrows():
        descriptor = DEFAULT_PRODUCT_REGISTRY.recognize(row["name"])

        if descriptor is None:
            continue
        if descriptor.product_role != "target_sensor":
            continue
        if descriptor.sensor_name not in required_sensors:
            continue

        recognized_rows.append({
            "remote_img": row["remote_path"],
            "parent": row["parent"],
            "img_name": row["name"],
            "img_bytes": int(row["bytes"]),
            "sensor": descriptor.sensor_name,
            "registry_key": descriptor.key,
        })

    recognized = pd.DataFrame(recognized_rows)

    if recognized.empty:
        raise RuntimeError(
            "No SpectralBridge-registered target-sensor ENVI products were found."
        )

    remote_object_paths = set(objects["remote_path"].astype(str))
    size_lookup = dict(zip(objects["remote_path"], objects["bytes"]))

    recognized["remote_hdr"] = recognized["remote_img"].map(
        lambda p: str(Path(p).with_suffix(".hdr"))
    )
    recognized["hdr_exists"] = recognized["remote_hdr"].isin(remote_object_paths)
    recognized["hdr_bytes"] = recognized["remote_hdr"].map(size_lookup)

    group_rows = []

    for parent, group in recognized.groupby("parent"):
        sensor_counts = group["sensor"].value_counts().to_dict()
        missing = [s for s in required_sensors if sensor_counts.get(s, 0) == 0]
        duplicates = [s for s in required_sensors if sensor_counts.get(s, 0) > 1]
        missing_hdr = group.loc[~group["hdr_exists"], "sensor"].tolist()

        complete = not missing and not duplicates and not missing_hdr

        group_rows.append({
            "flightline_dir": parent,
            "flightline_name": Path(parent).name,
            "registered_images": len(group),
            "complete": complete,
            "missing_sensors": "; ".join(missing),
            "duplicate_sensors": "; ".join(duplicates),
            "missing_headers": "; ".join(missing_hdr),
        })

    flightline_audit = pd.DataFrame(group_rows).sort_values(
        ["complete", "flightline_name"],
        ascending=[False, True],
    )

    flightline_audit.to_csv(FLIGHTLINE_AUDIT_CSV, index=False)

    complete_dirs = set(
        flightline_audit.loc[flightline_audit["complete"], "flightline_dir"]
    )

    complete_products = recognized[
        recognized["parent"].isin(complete_dirs)
    ].copy()

    if complete_products.empty:
        raise RuntimeError("No complete translation-ready flightlines were found.")

    transfer_rows = []

    for _, row in complete_products.iterrows():
        for kind, remote_path, size in (
            ("img", row["remote_img"], row["img_bytes"]),
            ("hdr", row["remote_hdr"], row["hdr_bytes"]),
        ):
            transfer_rows.append({
                "flightline_dir": row["parent"],
                "flightline_name": Path(row["parent"]).name,
                "sensor": row["sensor"],
                "registry_key": row["registry_key"],
                "kind": kind,
                "remote_path": remote_path,
                "bytes": int(size),
            })

    transfer_manifest = pd.DataFrame(transfer_rows).sort_values(
        ["flightline_name", "sensor", "kind", "remote_path"]
    ).reset_index(drop=True)

    expected_files_per_flightline = 2 * len(required_sensors)
    file_counts = transfer_manifest.groupby("flightline_dir").size()

    if not (file_counts == expected_files_per_flightline).all():
        raise RuntimeError(
            "Internal planning error: a complete flightline does not have "
            "exactly one IMG+HDR per required sensor."
        )

    transfer_manifest.to_csv(TRANSFER_MANIFEST_CSV, index=False)

    total_bytes = int(transfer_manifest["bytes"].sum())
    disk = shutil.disk_usage(BASE)
    free_bytes = int(disk.free)
    reserve_bytes = int(DISK_SAFETY_RESERVE_GB * 1024**3)
    stage_budget_bytes = min(
        int(free_bytes * MAX_STAGE_FRACTION_OF_FREE),
        max(0, free_bytes - reserve_bytes),
    )
    full_dataset_fits = total_bytes <= stage_budget_bytes

    plan = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "remote_source": REMOTE_SOURCE,
        "run_name": RUN_NAME,
        "spectralbridge_version": getattr(spectralbridge, "__version__", "unknown"),
        "git_commit": git_info["commit"],
        "translation_pairs": [pair.key for pair in translation_pairs],
        "required_sensors": required_sensors,
        "candidate_directories": len(flightline_audit),
        "complete_flightlines": int(flightline_audit["complete"].sum()),
        "incomplete_directories": int((~flightline_audit["complete"]).sum()),
        "transfer_objects": len(transfer_manifest),
        "transfer_bytes": total_bytes,
        "disk_free_bytes": free_bytes,
        "stage_budget_bytes": stage_budget_bytes,
        "full_dataset_fits": bool(full_dataset_fits),
        "summary_supported": SUMMARY_SUPPORTED,
    }

    write_json(PLAN_JSON, plan)

    print("Translation pairs:", len(plan["translation_pairs"]))
    print("Required sensors:", len(required_sensors))
    print("Candidate directories:", plan["candidate_directories"])
    print("Complete flightlines:", plan["complete_flightlines"])
    print("Incomplete directories:", plan["incomplete_directories"])
    print("Transfer objects:", plan["transfer_objects"])
    print(f"Transfer size: {total_bytes/1024**3:.2f} GB")
    print(f"Disk free: {free_bytes/1024**3:.2f} GB")
    print(f"Stage budget: {stage_budget_bytes/1024**3:.2f} GB")
    print("FULL_DATASET_FITS:", full_dataset_fits)
    print("SUMMARY_SUPPORTED:", SUMMARY_SUPPORTED)

    display(flightline_audit)

    if not full_dataset_fits:
        raise RuntimeError(
            "The complete eligible population does not fit safely. "
            "The notebook will not silently subsample."
        )

    print("\n✓ PLAN APPROVED.")
    print("Next: transfer will run automatically in RUN_STAGE='all' mode.")


## 7. TRANSFER stage

In [ ]:

if RUN_STAGE not in ("transfer", "all"):
    print("TRANSFER stage skipped.")
else:
    plan = read_json(require_file(PLAN_JSON))
    transfer_manifest = pd.read_csv(require_file(TRANSFER_MANIFEST_CSV))

    if plan["remote_source"] != REMOTE_SOURCE:
        raise RuntimeError("Persisted plan belongs to another remote source.")
    if not plan["full_dataset_fits"]:
        raise RuntimeError("Persisted plan did not pass the disk gate.")

    LOCAL_STAGE.mkdir(parents=True, exist_ok=True)

    write_json(
        LOCAL_STAGE / "_stage_provenance.json",
        {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "remote_source": REMOTE_SOURCE,
            "plan": plan,
            "git": git_info,
        },
    )

    transfer_manifest.to_csv(
        LOCAL_STAGE / "_selected_remote_manifest.csv",
        index=False,
    )

    failures = []
    reused = 0
    downloaded = 0

    for i, row in transfer_manifest.iterrows():
        remote_path = str(row["remote_path"])
        expected_bytes = int(row["bytes"])

        relative = Path(remote_path).relative_to(Path(REMOTE_SOURCE))
        local_path = LOCAL_STAGE / relative
        assert_within(local_path, LOCAL_STAGE)
        local_path.parent.mkdir(parents=True, exist_ok=True)

        if local_path.is_file() and local_path.stat().st_size == expected_bytes:
            reused += 1
            continue

        if local_path.exists():
            if local_path.is_file():
                local_path.unlink()
            else:
                raise RuntimeError(f"Unexpected local object: {local_path}")

        if i % 25 == 0:
            print(f"[{i+1}/{len(transfer_manifest)}] {relative}")

        res = run_cmd([
            GOCMD,
            "get",
            "--progress",
            "i:" + remote_path,
            str(local_path),
        ])

        if res.returncode != 0:
            failures.append({
                "remote_path": remote_path,
                "local_path": str(local_path),
                "returncode": res.returncode,
                "stderr": (res.stderr or "")[-1500:],
            })
        else:
            downloaded += 1

    failure_df = pd.DataFrame(failures)
    failure_df.to_csv(STATE_DIR / "transfer_failures.csv", index=False)

    if len(failure_df):
        display(failure_df.head(100))
        raise RuntimeError("One or more transfers failed.")

    validation_rows = []

    for _, row in transfer_manifest.iterrows():
        rel = Path(row["remote_path"]).relative_to(Path(REMOTE_SOURCE))
        local_path = LOCAL_STAGE / rel
        expected_bytes = int(row["bytes"])

        validation_rows.append({
            "relative_path": rel.as_posix(),
            "exists": local_path.is_file(),
            "expected_bytes": expected_bytes,
            "local_bytes": local_path.stat().st_size if local_path.is_file() else None,
            "size_match": (
                local_path.is_file()
                and local_path.stat().st_size == expected_bytes
            ),
        })

    stage_validation = pd.DataFrame(validation_rows)
    stage_validation.to_csv(STAGE_VALIDATION_CSV, index=False)

    if not stage_validation["size_match"].all():
        display(stage_validation[~stage_validation["size_match"]].head(100))
        raise RuntimeError("Staged archive failed exact size validation.")

    print("✓ TRANSFER APPROVED.")
    print("Reused:", reused)
    print("Downloaded:", downloaded)
    print("Objects validated:", len(stage_validation))
    print("Next: preflight will run automatically in RUN_STAGE='all' mode.")



## 8. RECONCILE stage: prove and remove duplicate scientific units

The raw CyVerse archive may contain the same scientific flightline in more than one
collection. That is not automatically a scientific problem, but it cannot be ignored.

This stage:

- uses the final flightline directory name as the initial duplicate key,
- finds repeated keys across the downloaded archive,
- aligns their required sensor products,
- requires matching file counts, registry keys, sensor labels, kinds, filenames, and
  byte sizes,
- computes SHA256 for repeated files when `HASH_DUPLICATE_GROUPS=True`,
- stops if any repeated copy differs,
- deterministically selects one representative source directory,
- builds `RECONCILED_STAGE` using hardlinks when possible and copies only when the
  filesystem cannot hardlink,
- writes a complete duplicate-resolution and reconciled-source manifest.

The raw downloaded stage remains untouched.


In [ ]:

if RUN_STAGE not in ("reconcile", "all"):
    print("RECONCILE stage skipped.")
else:
    plan = read_json(require_file(PLAN_JSON))
    transfer_manifest = pd.read_csv(require_file(TRANSFER_MANIFEST_CSV))
    stage_validation = pd.read_csv(require_file(STAGE_VALIDATION_CSV))

    if not bool(stage_validation["size_match"].all()):
        raise RuntimeError("Cannot reconcile because raw staged-file validation is not clean.")

    # One scientific-unit key per source directory. The current archive repeats some
    # NEON flightlines under multiple collection parents.
    source_units = (
        transfer_manifest[
            ["flightline_dir", "flightline_name"]
        ]
        .drop_duplicates()
        .sort_values(["flightline_name", "flightline_dir"])
        .reset_index(drop=True)
    )

    duplicate_counts = source_units["flightline_name"].value_counts()
    repeated_names = set(duplicate_counts[duplicate_counts > 1].index)

    print("Raw source directories:", len(source_units))
    print("Unique flightline names:", source_units["flightline_name"].nunique())
    print("Repeated flightline-name groups:", len(repeated_names))

    duplicate_rows = []
    selected_dirs = []

    for flightline_name, group in source_units.groupby("flightline_name", sort=True):
        dirs = sorted(group["flightline_dir"].astype(str).tolist())
        representative = dirs[0]

        if len(dirs) == 1:
            selected_dirs.append(representative)
            duplicate_rows.append({
                "flightline_name": flightline_name,
                "source_dir": representative,
                "representative_dir": representative,
                "group_size": 1,
                "status": "unique",
                "content_verified": True,
                "reason": "",
            })
            continue

        # Build comparable product inventories for each repeated copy.
        inventories = {}
        for d in dirs:
            subset = transfer_manifest[
                transfer_manifest["flightline_dir"].astype(str) == d
            ].copy()

            subset["basename"] = subset["remote_path"].map(lambda p: Path(str(p)).name)
            subset = subset.sort_values(
                ["sensor", "registry_key", "kind", "basename"]
            ).reset_index(drop=True)

            inventories[d] = subset

        ref = inventories[representative]
        compare_cols = [
            "sensor", "registry_key", "kind", "basename", "bytes"
        ]
        ref_signature = ref[compare_cols].astype(str).to_dict("records")

        group_ok = True
        reason = ""

        for d in dirs[1:]:
            other = inventories[d]
            other_signature = other[compare_cols].astype(str).to_dict("records")

            if other_signature != ref_signature:
                group_ok = False
                reason = "metadata_or_size_mismatch"
                break

        # Size/metadata identity is necessary but not sufficient. Hash only repeated
        # files to establish byte-for-byte equivalence.
        hash_records = {}

        if group_ok and HASH_DUPLICATE_GROUPS:
            for d in dirs:
                hashes = []
                for _, row in inventories[d].iterrows():
                    rel = Path(str(row["remote_path"])).relative_to(Path(REMOTE_SOURCE))
                    local_path = LOCAL_STAGE / rel
                    require_file(local_path)
                    hashes.append({
                        "sensor": str(row["sensor"]),
                        "registry_key": str(row["registry_key"]),
                        "kind": str(row["kind"]),
                        "basename": Path(str(row["remote_path"])).name,
                        "sha256": sha256_file(local_path),
                    })
                hash_records[d] = sorted(
                    hashes,
                    key=lambda x: (
                        x["sensor"], x["registry_key"], x["kind"], x["basename"]
                    ),
                )

            reference_hashes = hash_records[representative]
            for d in dirs[1:]:
                if hash_records[d] != reference_hashes:
                    group_ok = False
                    reason = "sha256_mismatch"
                    break

        status = "exact_duplicate" if group_ok else "conflicting_duplicate"

        for d in dirs:
            duplicate_rows.append({
                "flightline_name": flightline_name,
                "source_dir": d,
                "representative_dir": representative,
                "group_size": len(dirs),
                "status": status,
                "content_verified": bool(group_ok),
                "reason": reason,
            })

        if not group_ok:
            duplicate_df = pd.DataFrame(duplicate_rows)
            duplicate_df.to_csv(DUPLICATE_AUDIT_CSV, index=False)
            display(
                duplicate_df[
                    duplicate_df["flightline_name"] == flightline_name
                ]
            )
            raise RuntimeError(
                "Repeated scientific flightline copies are not identical. "
                f"Conflict: {flightline_name}. "
                "Do not choose a representative automatically."
            )

        selected_dirs.append(representative)

    duplicate_df = pd.DataFrame(duplicate_rows)
    duplicate_df.to_csv(DUPLICATE_AUDIT_CSV, index=False)

    selected_dirs = sorted(set(selected_dirs))
    reconciled_manifest = transfer_manifest[
        transfer_manifest["flightline_dir"].astype(str).isin(selected_dirs)
    ].copy()

    reconciled_manifest["source_relative_path"] = reconciled_manifest["remote_path"].map(
        lambda p: Path(str(p)).relative_to(Path(REMOTE_SOURCE)).as_posix()
    )

    # Use a clean deterministic layout, one directory per unique scientific flightline.
    reconciled_manifest["reconciled_relative_path"] = reconciled_manifest.apply(
        lambda row: (
            Path(str(row["flightline_name"]))
            / Path(str(row["remote_path"])).name
        ).as_posix(),
        axis=1,
    )

    if reconciled_manifest["reconciled_relative_path"].duplicated().any():
        dup = reconciled_manifest[
            reconciled_manifest["reconciled_relative_path"].duplicated(keep=False)
        ]
        display(dup)
        raise RuntimeError(
            "Reconciled layout would contain duplicate paths. "
            "Scientific-unit naming is not sufficiently unique."
        )

    reconciled_manifest.to_csv(RECONCILED_MANIFEST_CSV, index=False)

    # Rebuild reconciled stage only with explicit approval; never move raw sources.
    if RECONCILED_STAGE.exists():
        if not ALLOW_REBUILD_RECONCILED_STAGE:
            raise RuntimeError("Reconciled stage exists; resume from preflight or explicitly allow rebuilding it.")
        shutil.rmtree(RECONCILED_STAGE)
    RECONCILED_STAGE.mkdir(parents=True, exist_ok=True)

    link_count = 0
    copy_count = 0

    for _, row in reconciled_manifest.iterrows():
        raw_rel = Path(str(row["remote_path"])).relative_to(Path(REMOTE_SOURCE))
        src = LOCAL_STAGE / raw_rel
        dst = RECONCILED_STAGE / Path(row["reconciled_relative_path"])

        require_file(src)
        assert_within(dst, RECONCILED_STAGE)
        dst.parent.mkdir(parents=True, exist_ok=True)

        try:
            os.link(src, dst)
            link_count += 1
        except OSError:
            shutil.copy2(src, dst)
            copy_count += 1

        if dst.stat().st_size != src.stat().st_size:
            raise RuntimeError(f"Reconciled file-size mismatch: {dst}")

    # Preserve reconciliation evidence inside the stage without creating files that
    # match SpectralBridge product patterns.
    write_json(
        RECONCILED_STAGE / "_reconciliation_provenance.json",
        {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "raw_stage": str(LOCAL_STAGE),
            "reconciled_stage": str(RECONCILED_STAGE),
            "raw_source_directories": int(len(source_units)),
            "unique_scientific_units": int(len(selected_dirs)),
            "repeated_groups": int(len(repeated_names)),
            "hash_duplicate_groups": bool(HASH_DUPLICATE_GROUPS),
            "hardlinks_created": int(link_count),
            "copies_created": int(copy_count),
            "git": git_info,
        },
    )

    exact_duplicate_groups = int(
        duplicate_df.loc[
            duplicate_df["status"] == "exact_duplicate",
            "flightline_name",
        ].nunique()
    )

    reconciliation = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "raw_source_directories": int(len(source_units)),
        "unique_scientific_units": int(len(selected_dirs)),
        "duplicate_groups": int(len(repeated_names)),
        "exact_duplicate_groups": exact_duplicate_groups,
        "conflicting_duplicate_groups": int(
            duplicate_df.loc[
                duplicate_df["status"] == "conflicting_duplicate",
                "flightline_name",
            ].nunique()
        ),
        "reconciled_objects": int(len(reconciled_manifest)),
        "hardlinks_created": int(link_count),
        "copies_created": int(copy_count),
        "reconciled_stage": str(RECONCILED_STAGE),
        "passed": True,
    }

    write_json(RECONCILIATION_JSON, reconciliation)

    print("✓ RECONCILIATION APPROVED.")
    print("Raw source directories:", reconciliation["raw_source_directories"])
    print("Unique scientific units:", reconciliation["unique_scientific_units"])
    print("Exact duplicate groups:", reconciliation["exact_duplicate_groups"])
    print("Reconciled objects:", reconciliation["reconciled_objects"])
    print("Hardlinks:", reconciliation["hardlinks_created"])
    print("Copies:", reconciliation["copies_created"])
    display(duplicate_df)
    print("Next: preflight will run automatically in RUN_STAGE='all' mode.")



## Preflight semantics used by v7

`preflight_only=True` intentionally does not execute pixel extraction or
regression fitting. Therefore `translation_pair_count` and `regression_count`
may both remain zero even when all requested translation pairs are valid and
available.

The correct preflight contract is based on the pair names exposed by the
preflight metadata:

- `selected_translation_pairs`
- `preflight.available_translation_pairs`

The numerical `translation_pair_count` becomes meaningful in the full analysis
result, after translation regressions have actually run.


## 9. PREFLIGHT stage

In [ ]:

if RUN_STAGE not in ("preflight", "all"):
    print("PREFLIGHT stage skipped.")
else:
    plan = read_json(require_file(PLAN_JSON))
    reconciliation = read_json(require_file(RECONCILIATION_JSON))

    if not reconciliation.get("passed"):
        raise RuntimeError("Reconciliation gate did not pass.")

    TEMP_DIR.mkdir(parents=True, exist_ok=True)

    result = run_bulk_pipeline(
        RECONCILED_STAGE,
        PREFLIGHT_OUTPUT,
        input_mode="flightline_outputs",
        analysis="translation",
        on_invalid="exclude",
        materialize_observations=False,
        preflight_only=True,
        extraction_workers=EXTRACTION_WORKERS,
        temp_directory=TEMP_DIR,
    )

    planned_pairs = sorted(plan["translation_pairs"])
    selected_pairs = sorted(result.get("selected_translation_pairs") or [])
    preflight_meta = result.get("preflight") or {}
    available_pairs = sorted(
        preflight_meta.get("available_translation_pairs") or []
    )

    # In preflight-only mode, translation_pair_count can legitimately be zero
    # because no regressions have been executed yet. Pair availability is
    # validated by pair names, not by executed-regression count.
    gates = {
        "accepted_population_matches_reconciled_stage": (
            int(result["accepted_flightline_count"])
            == int(reconciliation["unique_scientific_units"])
        ),
        "no_duplicates_after_reconciliation": (
            int(result["duplicate_count"]) == 0
        ),
        "no_rejected_sources": (
            int(result["rejected_source_count"]) == 0
        ),
        "selected_pairs_match_plan": (
            selected_pairs == planned_pairs
        ),
        "available_pairs_match_plan": (
            available_pairs == planned_pairs
        ),
        "no_materialized_observations": (
            result.get("materialized_observations") is None
        ),
        "preflight_did_not_scan_observations": (
            int(result.get("row_count", 0)) == 0
        ),
    }

    baseline = {
        "accepted_flightline_count": int(result["accepted_flightline_count"]),
        "duplicate_count": int(result["duplicate_count"]),
        "rejected_source_count": int(result["rejected_source_count"]),
        "planned_translation_pairs": planned_pairs,
        "selected_translation_pairs": selected_pairs,
        "available_translation_pairs": available_pairs,
        "expected_translation_pair_count_after_analysis": len(planned_pairs),
        "input_stage": str(RECONCILED_STAGE),
    }

    record = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "passed": bool(all(gates.values())),
        "gates": gates,
        "result": result,
        "authoritative_baseline": baseline,
        "preflight_semantics": {
            "translation_pair_count_is_executed_count": True,
            "preflight_translation_pair_count": int(
                result.get("translation_pair_count", 0)
            ),
            "preflight_regression_count": int(
                result.get("regression_count", 0)
            ),
            "pair_availability_validated_by_names": True,
        },
    }

    write_json(PREFLIGHT_GATE_JSON, record)

    display(
        pd.DataFrame(
            [{"gate": k, "passed": v} for k, v in gates.items()]
        )
    )

    print("Planned pairs:")
    for x in planned_pairs:
        print(" -", x)

    print("\nAvailable in SpectralBridge preflight:")
    for x in available_pairs:
        print(" -", x)

    print(
        "\nPreflight translation_pair_count:",
        result.get("translation_pair_count"),
        "(expected to be 0 before regressions run)"
    )

    if not record["passed"]:
        print(json.dumps(result, indent=2, default=str)[:20000])
        raise RuntimeError(
            "SpectralBridge preflight did not approve the reconciled "
            "scientific population or requested translation-pair set."
        )

    print("\n✓ PREFLIGHT APPROVED.")
    print("Accepted scientific units:", baseline["accepted_flightline_count"])
    print(
        "Translation pairs available:",
        len(baseline["available_translation_pairs"]),
    )
    print("Next: analysis will run automatically in RUN_STAGE='all' mode.")


## 10. ANALYZE stage

This is the expensive population analysis.

### Important production semantics

`translation_pair_count` is the number of named registry translation relationships.
It is **not** the number of band-level coefficient rows. The production run can
therefore have 4 named translation relationships while yielding 18 source-target
band relationships and 54 candidate coefficients across three weighting schemes.

The gate below validates the accepted population, selected pair identities,
execution of regressions, and the complete compact-output contract. It does not
repeat the v7 mistake of equating registry relation count with coefficient-row count.

The authoritative compact outputs include the original candidate coefficients,
per-flightline and per-site fits, LOSO results, sufficient statistics, and manifest.
These are required before the run may proceed.


In [ ]:

if RUN_STAGE not in ("analyze", "all"):
    print("ANALYZE stage skipped.")
else:
    preflight_gate = read_json(require_file(PREFLIGHT_GATE_JSON))

    if not preflight_gate.get("passed"):
        raise RuntimeError("Persisted preflight did not pass.")

    baseline = preflight_gate["authoritative_baseline"]
    expected_pairs = sorted(baseline["available_translation_pairs"])
    expected_pair_count = int(
        baseline["expected_translation_pair_count_after_analysis"]
    )

    TEMP_DIR.mkdir(parents=True, exist_ok=True)

    result = run_bulk_pipeline(
        RECONCILED_STAGE,
        FINAL_OUTPUT,
        input_mode="flightline_outputs",
        analysis="translation",
        on_invalid="exclude",
        materialize_observations=False,
        preflight_only=False,
        extraction_workers=EXTRACTION_WORKERS,
        temp_directory=TEMP_DIR,
    )

    final_selected_pairs = sorted(
        result.get("selected_translation_pairs") or []
    )

    gates = {
        "accepted_population_matches_preflight": (
            int(result["accepted_flightline_count"])
            == int(baseline["accepted_flightline_count"])
        ),
        "duplicate_count_matches_preflight": (
            int(result["duplicate_count"])
            == int(baseline["duplicate_count"])
        ),
        "rejected_source_count_matches_preflight": (
            int(result["rejected_source_count"])
            == int(baseline["rejected_source_count"])
        ),
        "selected_pair_names_match_preflight": (
            final_selected_pairs == expected_pairs
        ),
        # translation_pair_count is a registry-level relation count.  It is
        # intentionally NOT compared with the number of band-level coefficient
        # rows.  That mistaken semantic assumption is what stopped the original
        # production notebook after the expensive analysis had already succeeded.
        "translation_pairs_are_nonempty": (
            int(result.get("translation_pair_count", 0)) > 0
        ),
        "regressions_were_executed": (
            int(result.get("regression_count", 0)) > 0
        ),
        "no_materialized_observations": (
            result.get("materialized_observations") is None
        ),
    }

    compact_required = [
        "catalog/bulk_manifest.json",
        "coefficients/candidate_translation_coefficients.parquet",
        "coefficients/candidate_translation_coefficients.json",
        "analyses/sensor_translation/per_flightline.parquet",
        "analyses/sensor_translation/per_site.parquet",
        "analyses/sensor_translation/flightline_balanced.parquet",
        "analyses/sensor_translation/site_balanced.parquet",
        "analyses/leave_one_site_out/leave_one_site_out.parquet",
        "statistics/translation_sufficient_statistics.parquet",
    ]

    missing = [
        rel
        for rel in compact_required
        if not (FINAL_OUTPUT / rel).is_file()
        or (FINAL_OUTPUT / rel).stat().st_size == 0
    ]

    gates["compact_output_contract"] = not missing

    record = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "passed": bool(all(gates.values())),
        "gates": gates,
        "missing_outputs": missing,
        "result": result,
        "compact_required": compact_required,
        "preflight_baseline": baseline,
    }

    write_json(ANALYSIS_GATE_JSON, record)

    display(
        pd.DataFrame(
            [{"gate": k, "passed": v} for k, v in gates.items()]
        )
    )

    print("Final translation_pair_count:", result["translation_pair_count"])
    print("Final regression_count:", result.get("regression_count"))

    if not record["passed"]:
        print("Missing outputs:", missing)
        print(json.dumps(result, indent=2, default=str)[:20000])
        raise RuntimeError("Final bulk analysis failed production gates.")

    print("\n✓ ANALYSIS APPROVED.")
    print("Accepted scientific units:", result["accepted_flightline_count"])
    print("Translation pairs executed:", result["translation_pair_count"])
    print("Regressions executed:", result.get("regression_count"))

    if SUMMARY_SUPPORTED:
        print("Next: summarization will run automatically in RUN_STAGE='all' mode.")
    else:
        print("This checkout has no compatible compact summary API.")
        print("The summary stage will record that limitation and continue.")


## 11. SUMMARIZE AND INTERPRET stage

This stage uses the **current public SpectralBridge API**:

```python
from spectralbridge import summarize_bulk_results
from spectralbridge.bulk import BulkResultsConfig
```

It operates only on the compact outputs from the completed bulk run. It must not
reopen source rasters or regenerate the expensive population statistics.

This stage is mandatory. It produces and validates:

- pair-band summary
- weighting comparison
- flightline stability
- site stability
- leave-one-site-out transferability
- attention flags
- machine-readable bulk-results summary
- one-page QA dashboard PNG and PDF
- detailed diagnostic figures
- three publication figures
- Markdown report
- PDF report

Failure to generate any required product is a production failure.


In [ ]:

if RUN_STAGE not in ("summarize", "all"):
    print("SUMMARIZE stage skipped.")
else:
    analysis_gate = read_json(require_file(ANALYSIS_GATE_JSON))
    if not analysis_gate.get("passed"):
        raise RuntimeError("Persisted analysis gate did not pass.")

    # Current SpectralBridge API. This reads only compact completed-run products.
    summary = summarize_bulk_results(
        FINAL_OUTPUT,
        config=BulkResultsConfig(),
        make_figures=True,
        make_report=True,
    )

    required = [
        "analyses/bulk_results/pair_band_summary.parquet",
        "analyses/bulk_results/weighting_comparison.parquet",
        "analyses/bulk_results/flightline_stability.parquet",
        "analyses/bulk_results/site_stability.parquet",
        "analyses/bulk_results/loso_transferability.parquet",
        "analyses/bulk_results/attention_flags.parquet",
        "analyses/bulk_results/bulk_results_summary.json",

        "figures/bulk_results/summary/bulk_qa_summary.png",
        "figures/bulk_results/summary/bulk_qa_summary.pdf",

        "figures/bulk_results/diagnostics/translation_coefficients_and_fit.png",
        "figures/bulk_results/diagnostics/fitted_correction_magnitude.png",
        "figures/bulk_results/diagnostics/stability_and_transferability.png",

        "figures/bulk_results/publication/translation_performance.png",
        "figures/bulk_results/publication/translation_performance.pdf",
        "figures/bulk_results/publication/translation_stability.png",
        "figures/bulk_results/publication/translation_stability.pdf",
        "figures/bulk_results/publication/generalization_and_failures.png",
        "figures/bulk_results/publication/generalization_and_failures.pdf",

        "reports/bulk_results/bulk_translation_results.md",
        "reports/bulk_results/bulk_translation_results.pdf",
    ]

    missing = [
        rel for rel in required
        if not (FINAL_OUTPUT / rel).is_file()
        or (FINAL_OUTPUT / rel).stat().st_size == 0
    ]

    # Basic scientific sanity checks on the compact interpretation.
    import pyarrow.parquet as pq

    pair_summary = pq.read_table(
        FINAL_OUTPUT / "analyses/bulk_results/pair_band_summary.parquet"
    )
    weighting = pq.read_table(
        FINAL_OUTPUT / "analyses/bulk_results/weighting_comparison.parquet"
    )
    flags = pq.read_table(
        FINAL_OUTPUT / "analyses/bulk_results/attention_flags.parquet"
    )

    scientific_gates = {
        "required_summary_products_exist": not missing,
        "pair_band_summary_nonempty": pair_summary.num_rows > 0,
        "weighting_comparison_nonempty": weighting.num_rows > 0,
        "weighting_has_multiple_levels": (
            "analysis_level" in weighting.column_names
            and len(set(weighting["analysis_level"].to_pylist())) >= 3
        ),
        "attention_flags_table_readable": flags.num_rows >= 0,
    }

    record = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "passed": bool(all(scientific_gates.values())),
        "gates": scientific_gates,
        "missing_outputs": missing,
        "required": required,
        "summary_return": summary,
        "pair_band_rows": pair_summary.num_rows,
        "weighting_rows": weighting.num_rows,
        "attention_flag_rows": flags.num_rows,
    }
    write_json(SUMMARY_GATE_JSON, record)

    display(pd.DataFrame(
        [{"gate": k, "passed": v} for k, v in scientific_gates.items()]
    ))

    if not record["passed"]:
        print("Missing summary/report products:", missing)
        raise RuntimeError(
            "Bulk scientific summary / QA / reporting stage failed production gates."
        )

    print("✓ SCIENTIFIC SUMMARY / QA / REPORTS APPROVED")
    print("Pair-band rows:", pair_summary.num_rows)
    print("Weighting rows:", weighting.num_rows)
    print("Attention flags:", flags.num_rows)


## 12. PACKAGE stage

Build one self-contained closeout package.

Unlike the old v9 analysis-only package, this package **must include the original
compact production artifacts**, especially:

- `coefficients/candidate_translation_coefficients.parquet`
- `coefficients/candidate_translation_coefficients.json`
- `statistics/translation_sufficient_statistics.parquet`
- per-flightline, per-site, balanced, and LOSO analysis tables
- manifest
- downstream bulk QA tables, figures, and reports
- state/gate/provenance records

This prevents the exact artifact-loss problem encountered during the first production run.


In [ ]:

if RUN_STAGE not in ("package", "all"):
    print("PACKAGE stage skipped.")
else:
    analysis_gate = read_json(require_file(ANALYSIS_GATE_JSON))
    if not analysis_gate.get("passed"):
        raise RuntimeError("Persisted analysis gate did not pass.")

    summary_gate = read_json(require_file(SUMMARY_GATE_JSON))
    if not summary_gate.get("passed"):
        raise RuntimeError("Persisted scientific summary gate did not pass.")

    # Rebuild the package only with explicit approval. FINAL_OUTPUT remains untouched.
    if PACKAGE_DIR.exists():
        if not ALLOW_REBUILD_PACKAGE:
            raise RuntimeError("Closeout package exists; explicitly allow rebuilding it before replacing it.")
        shutil.rmtree(PACKAGE_DIR)

    results_root = PACKAGE_DIR / "results"
    provenance_root = PACKAGE_DIR / "provenance"
    results_root.mkdir(parents=True, exist_ok=True)
    provenance_root.mkdir(parents=True, exist_ok=True)

    authoritative_paths = [
        "catalog/bulk_manifest.json",
        "coefficients/candidate_translation_coefficients.parquet",
        "coefficients/candidate_translation_coefficients.json",
        "analyses/sensor_translation/per_flightline.parquet",
        "analyses/sensor_translation/per_site.parquet",
        "analyses/sensor_translation/flightline_balanced.parquet",
        "analyses/sensor_translation/site_balanced.parquet",
        "analyses/leave_one_site_out/leave_one_site_out.parquet",
        "statistics/translation_sufficient_statistics.parquet",
    ]

    for rel in authoritative_paths:
        src = require_file(FINAL_OUTPUT / rel)
        dst = results_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    # Copy downstream scientific interpretation if supported/generated.
    for rel_root in [
        "analyses/bulk_results",
        "figures/bulk_results",
        "reports/bulk_results",
    ]:
        src_root = FINAL_OUTPUT / rel_root
        if not src_root.is_dir():
            continue
        for src in src_root.rglob("*"):
            if src.is_file():
                rel = src.relative_to(FINAL_OUTPUT)
                dst = results_root / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)

    # Preserve the state needed to audit/restart the production run.
    for src in sorted(STATE_DIR.glob("*")):
        if src.is_file():
            shutil.copy2(src, provenance_root / src.name)

    run_metadata = {
        "schema_version": 2,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "run_name": RUN_NAME,
        "remote_source": REMOTE_SOURCE,
        "local_stage": str(LOCAL_STAGE),
        "reconciled_stage": str(RECONCILED_STAGE),
        "final_output": str(FINAL_OUTPUT),
        "spectralbridge_version": getattr(spectralbridge, "__version__", "unknown"),
        "authoritative_compact_artifacts": authoritative_paths,
    }
    write_json(provenance_root / "production_metadata.json", run_metadata)

    manifest_rows = []
    for p in sorted(PACKAGE_DIR.rglob("*")):
        if p.is_file():
            manifest_rows.append({
                "relative_path": p.relative_to(PACKAGE_DIR).as_posix(),
                "bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            })

    manifest = {
        "schema_version": 2,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "file_count": len(manifest_rows),
        "total_bytes": sum(x["bytes"] for x in manifest_rows),
        "files": manifest_rows,
    }
    write_json(PACKAGE_DIR / "PACKAGE_MANIFEST.json", manifest)

    # Final hard gate: prove the coefficient artifact is in BOTH the production
    # output and the closeout package before allowing upload.
    required_packaged = [
        results_root / rel for rel in authoritative_paths
    ]
    missing_packaged = [
        str(p) for p in required_packaged
        if not p.is_file() or p.stat().st_size == 0
    ]
    if missing_packaged:
        raise RuntimeError(
            "Closeout package is missing authoritative compact artifacts: "
            + ", ".join(missing_packaged)
        )

    record = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "passed": True,
        "package_dir": str(PACKAGE_DIR),
        "file_count": manifest["file_count"],
        "total_bytes": manifest["total_bytes"],
        "authoritative_artifacts_packaged": True,
    }
    write_json(PACKAGE_GATE_JSON, record)

    print("✓ COMPLETE CLOSEOUT PACKAGE APPROVED")
    print("Package:", PACKAGE_DIR)
    print("Files:", manifest["file_count"])
    print(f"Size: {manifest['total_bytes']/1024**2:.1f} MB")
    print("Original candidate coefficients are included.")


## 13. UPLOAD stage

Remote writing is intentionally opt-in. Set `UPLOAD_RESULTS = True` in the
configuration only after inspecting the package gate. The uploaded object is the
complete closeout package, including the original coefficient artifacts.


In [ ]:

if RUN_STAGE not in ("upload", "all") or not UPLOAD_RESULTS:
    print("UPLOAD stage skipped. Set UPLOAD_RESULTS=True to enable the remote write.")
else:
    package_gate = read_json(require_file(PACKAGE_GATE_JSON))

    if not package_gate.get("passed"):
        raise RuntimeError("Persisted package QA gate did not pass.")

    require_dir(PACKAGE_DIR)

    expected_remote_folder = REMOTE_SOURCE.rstrip("/") + "/" + PACKAGE_DIR.name
    expected_remote_uri = "i:" + expected_remote_folder

    parent_probe = run_cmd([GOCMD, "ls", REMOTE_SOURCE_URI])
    if parent_probe.returncode != 0:
        raise RuntimeError("Remote source collection is no longer accessible.")

    existing = run_cmd([GOCMD, "ls", expected_remote_uri])
    if existing.returncode == 0:
        raise RuntimeError(
            "A remote result folder with this exact name already exists. "
            "The notebook will not overwrite it."
        )

    print("Uploading:")
    print(PACKAGE_DIR)
    print("Into:")
    print(REMOTE_SOURCE_URI)

    res = run_cmd([
        GOCMD,
        "put",
        "--progress",
        str(PACKAGE_DIR),
        REMOTE_SOURCE_URI,
    ])

    print("Return code:", res.returncode)
    print((res.stdout or "")[-4000:])
    print((res.stderr or "")[-4000:])

    record = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "returncode": res.returncode,
        "remote_parent": REMOTE_SOURCE,
        "expected_remote_folder": expected_remote_folder,
        "passed": res.returncode == 0,
    }

    write_json(UPLOAD_GATE_JSON, record)

    if res.returncode != 0:
        raise RuntimeError("Direct result-folder upload failed.")

    print("✓ UPLOAD COMMAND SUCCEEDED.")
    print("Next: verification will run automatically in RUN_STAGE='all' mode.")


## 14. VERIFY stage

In [ ]:

if RUN_STAGE not in ("verify", "all") or not UPLOAD_RESULTS:
    print("Remote VERIFY skipped because upload is disabled.")
else:
    upload_gate = read_json(require_file(UPLOAD_GATE_JSON))

    if not upload_gate.get("passed"):
        raise RuntimeError("Persisted upload gate did not pass.")

    expected_remote_folder = upload_gate["expected_remote_folder"]
    expected_remote_uri = "i:" + expected_remote_folder

    targets = [
        expected_remote_uri,
        expected_remote_uri + "/results",
        expected_remote_uri + "/preflight",
        expected_remote_uri + "/provenance",
        expected_remote_uri + "/PACKAGE_MANIFEST.json",
    ]

    rows = []

    for target in targets:
        res = run_cmd([GOCMD, "ls", target])
        rows.append({
            "target": target,
            "accessible": res.returncode == 0,
            "preview": (res.stdout or "").replace("\n", " | ")[:700],
            "error": (res.stderr or "")[:300],
        })

    verify_df = pd.DataFrame(rows)
    display(verify_df)

    if not verify_df["accessible"].all():
        raise RuntimeError(
            "Upload succeeded, but one or more required remote result targets "
            "could not be verified."
        )

    print("✓ COMPLETE BULK PACKAGE VERIFIED ON CYVERSE")
    print(expected_remote_uri)
    print("\nLocal staged data and results were NOT deleted.")


## 15. FINAL LOCAL AUDIT AND PRODUCTION VERDICT

The notebook may declare success only when both the authoritative compact analysis
artifacts **and** the scientific interpretation/report products are preserved in the
closeout package.

Authoritative compact products are compared byte-for-byte by SHA256. Summary,
figure, and report products must exist and be nonempty in both production output
and package.


In [ ]:
if RUN_STAGE not in ("all", "package", "verify"):
    print("Final package audit skipped for this stage.")
else:
    
    analysis_gate = read_json(require_file(ANALYSIS_GATE_JSON))
    summary_gate = read_json(require_file(SUMMARY_GATE_JSON))
    package_gate = read_json(require_file(PACKAGE_GATE_JSON))
    
    for name, gate in [
        ("analysis", analysis_gate),
        ("summary", summary_gate),
        ("package", package_gate),
    ]:
        if not gate.get("passed"):
            raise RuntimeError(f"{name.upper()} gate did not pass.")
    
    authoritative = [
        "catalog/bulk_manifest.json",
        "coefficients/candidate_translation_coefficients.parquet",
        "coefficients/candidate_translation_coefficients.json",
        "statistics/translation_sufficient_statistics.parquet",
        "analyses/sensor_translation/per_flightline.parquet",
        "analyses/sensor_translation/per_site.parquet",
        "analyses/sensor_translation/flightline_balanced.parquet",
        "analyses/sensor_translation/site_balanced.parquet",
        "analyses/leave_one_site_out/leave_one_site_out.parquet",
    ]
    
    derived_required = list(summary_gate["required"])
    
    audit_rows = []
    for rel in authoritative:
        prod = FINAL_OUTPUT / rel
        packed = PACKAGE_DIR / "results" / rel
        audit_rows.append({
            "class": "authoritative",
            "artifact": rel,
            "production_exists": prod.is_file() and prod.stat().st_size > 0,
            "package_exists": packed.is_file() and packed.stat().st_size > 0,
            "same_sha256": (
                prod.is_file() and packed.is_file()
                and sha256_file(prod) == sha256_file(packed)
            ),
        })
    
    for rel in derived_required:
        prod = FINAL_OUTPUT / rel
        packed = PACKAGE_DIR / "results" / rel
        audit_rows.append({
            "class": "derived_QA_report",
            "artifact": rel,
            "production_exists": prod.is_file() and prod.stat().st_size > 0,
            "package_exists": packed.is_file() and packed.stat().st_size > 0,
            "same_sha256": (
                prod.is_file() and packed.is_file()
                and sha256_file(prod) == sha256_file(packed)
            ),
        })
    
    final_audit = pd.DataFrame(audit_rows)
    display(final_audit)
    
    checks = ["production_exists", "package_exists", "same_sha256"]
    if not final_audit[checks].all().all():
        failed = final_audit.loc[~final_audit[checks].all(axis=1)]
        display(failed)
        raise RuntimeError("FINAL LOCAL AUDIT FAILED.")
    
    print("")
    print("=" * 72)
    print("✓ FULL SPECTRALBRIDGE BULK PRODUCTION WORKFLOW PASSED")
    print("=" * 72)
    print("Analysis gate: PASSED")
    print("Scientific summary / QA / reports: PASSED")
    print("Closeout package: PASSED")
    print("SHA256 preservation audit: PASSED")
    if UPLOAD_RESULTS:
        print("CyVerse upload/verification: requested; inspect upload gate above.")
    else:
        print("CyVerse upload: intentionally disabled")
    print("")
    print("Production output:", FINAL_OUTPUT)
    print("Closeout package:", PACKAGE_DIR)


# Operating instructions

## Fresh production run

1. Use a current checkout of `earthlab/spectralbridge`.
2. Set `REMOTE_SOURCE` to any curated CyVerse collection with the same completed-flightline product structure; review `BASE`, output paths, and disk limits.
3. Set `RUN_STAGE = "all"`.
4. Set `RUN = True`. Leave `UPLOAD_RESULTS = False` for the first production pass.
5. Use **Run All**.
6. Do not consider the run successful unless the final cell prints:

   **FULL SPECTRALBRIDGE BULK PRODUCTION WORKFLOW PASSED**

7. Inspect the generated dashboard, diagnostics, publication figures, Markdown
   report, and PDF report.
8. If the remote closeout copy is desired, set `UPLOAD_RESULTS = True` and run
   the upload/verify stages.

## Resume an already-completed expensive analysis

If `FINAL_OUTPUT` already contains a valid compact production run, do **not**
recompute the population analysis merely to test reporting. Set:

```python
RUN_STAGE = "summarize"
```

run the summary stage, then run package and final audit. Existing reconciled or closeout directories are never deleted without explicitly setting the corresponding `ALLOW_REBUILD_*` flag. Reuse completed analysis outputs instead of rerunning the population scan. The current
`summarize_bulk_results()` path operates only on compact completed-run outputs.

The notebook intentionally fails on stale or missing public APIs. Compatibility
fallbacks are not allowed in the production workflow.
